In [1]:
import tiktoken
import torch
import gc
import time

tokenizer = tiktoken.get_encoding("gpt2")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from SLM.GPT import GPTModel
from SLM.LoadModel import load_weights_into_gpt

print(device)
batch_size = 1
size = "355M"
model = GPTModel(size)

load_weights_into_gpt(model, size)
model.half()
model = model.to(device)
# checkpoint = torch.load(f"Checkpoint/summary1.pth", weights_only=True)
# model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

cuda


GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [2]:
for param in model.parameters():
    param.requires_grad = False

for param in model.trf_blocks[2].parameters():
    param.requires_grad = True

In [3]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad == True)
print(f"Total number of trainable parameters: {total_trainable:,} approx {(total_trainable/total_params)*100:.2f}%")


Total number of parameters: 406,286,336
Total number of trainable parameters: 12,596,224 approx 3.10%


In [4]:
import json

file_path = "summaryWithContext.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Number of entries:", len(data))

import re


def clean_text(text):
    """
    Cleans the input text by removing unwanted patterns, special characters, and normalizing whitespace.
    """

    # Remove pronunciation guides (e.g., /ˈkɑːmələ ˈdeɪvi/)
    text = re.sub(r"\/.*?\/|\(.*?\)", "", text)

    # Remove special characters (e.g., ;, -, etc.)
    # text = re.sub(r'[;,\-()]', ' ', text)

    # Remove square brackets shit
    text = re.sub(r"\[.*?\]", "", text)

    # Normalize whitespace
    # text = re.sub(r'\s+', ' ', text).strip()

    # Remove Unicode characters (e.g., \u02c8)
    text = re.sub(r"\\u[0-9a-fA-F]{4}", "", text)

    # Remove dates (e.g., October 20, 1964)
    # text = re.sub(r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b', '', text)

    # Remove titles (e.g., Dr., Mr., Ms., etc.)
    text = re.sub(r"\b(?:Dr|Mr|Ms|Mrs|Prof)\.\s*", "", text)

    # Remove newlines (\n) and replace with a space
    # text = re.sub(r'\n', ' ', text)

    # Normalize whitespace again
    # text = re.sub(r'\s+', ' ', text).strip()

    return text


def format_input(entry):
    instruction_text = (
        f"Below is a question and a context. "
        f"Your task is to generate an accurate response based on the provided context."
        f"\n\n### Question:\n{entry['question']}"
    )

    input_text = f"\n\n### Context:\n{entry['context']}" if entry["context"] else ""

    return instruction_text + input_text


from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{str(entry['response'])}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


def custom_collate_fn(
    batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item) + 1 for item in batch)

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # New: Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # New: Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor


from functools import partial

customized_collate_fn = partial(
    custom_collate_fn, device=device, allowed_max_length=1024
)

train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)  # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion : train_portion + test_portion]
val_data = data[train_portion + test_portion :]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))


from torch.utils.data import DataLoader


num_workers = 0


torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers,
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate(
    model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        idx_cond = idx_cond.to(device)
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val, torch.tensor(float("-inf")).to(device), logits
            )

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if (
            idx_next == eos_id
        ):  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx.to(device), idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dimension
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # remove batch dimension
    return tokenizer.decode(flat.tolist())


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    model = model.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

Number of entries: 1823
Training set length: 1549
Validation set length: 92
Test set length: 182


In [5]:
from torch.amp import GradScaler, autocast

scaler = GradScaler()


def train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs,
    eval_freq,
    eval_iter,
    tokenizer,
):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    global_step = 0
    gc.collect()
    torch.cuda.empty_cache()
    model = model.to(device)

    for epoch in range(num_epochs):
        gc.collect()
        torch.cuda.empty_cache()
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            with autocast(device_type="cuda", dtype=torch.float16):
                logits = model(input_batch)
                loss = torch.nn.functional.cross_entropy(
                    logits.flatten(0, 1), target_batch.flatten()
                )
            # loss.backward()
            # optimizer.step()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            global_step += 1
            tokens_seen += input_batch.numel()

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                print(
                    f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}"
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

    return train_losses, val_losses, track_tokens_seen

In [6]:
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, eps=1e-4, weight_decay=0.1, fused=True
)
num_epochs = 1
train_losses, val_losses, tokens_seen = train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=5,
    tokenizer=tokenizer,
)
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000005): Train loss 2.726, Val loss 2.886
Ep 1 (Step 000010): Train loss 2.520, Val loss 2.724
Ep 1 (Step 000015): Train loss 2.453, Val loss 2.793
Ep 1 (Step 000020): Train loss 2.519, Val loss 2.784
Ep 1 (Step 000025): Train loss 2.577, Val loss 2.725
Ep 1 (Step 000030): Train loss 2.355, Val loss 2.596
Ep 1 (Step 000035): Train loss 2.478, Val loss 2.568
Ep 1 (Step 000040): Train loss 2.384, Val loss 2.801
Ep 1 (Step 000045): Train loss 2.123, Val loss 2.579
Ep 1 (Step 000050): Train loss 2.186, Val loss 2.550
Ep 1 (Step 000055): Train loss 2.032, Val loss 2.579
Ep 1 (Step 000060): Train loss 2.321, Val loss 2.526
Ep 1 (Step 000065): Train loss 2.173, Val loss 2.498
Ep 1 (Step 000070): Train loss 2.159, Val loss 2.521
Ep 1 (Step 000075): Train loss 2.501, Val loss 2.509
Ep 1 (Step 000080): Train loss 2.280, Val loss 2.496
Ep 1 (Step 000085): Train loss 2.188, Val loss 2.507
Ep 1 (Step 000090): Train loss 2.233, Val loss 2.493
Ep 1 (Step 000095): Train loss 2.188, Val loss

KeyboardInterrupt: 

In [30]:
entry = {
    "question": "who is messi?",
    "context": "Lionel Andrés Leo Messi  is an Argentine professional footballer who plays as a forward for and captains both Major League Soccer club Inter Miami and the Argentina national team. Widely regarded as one of the greatest players of all time, Messi set numerous records for individual accolades won throughout his professional footballing career such as eight Ballon d'Or awards and four the Best FIFA Men's Player awards. He is the most decorated player in the history of professional football having won 45 team trophies, including twelve league titles, four UEFA Champions Leagues, two Copa Américas, and one FIFA World Cup. Messi holds the records for most European Golden Shoes , most goals in a calendar year , most goals for a single club , most goals , hat-tricks  and assists  in La Liga, most assists  and goal contributions  in the Copa América, most goal contributions  in the World Cup, most international appearances  and international goals  by a South American male, and the second-most in the latter category outright.",
}

# entry = {
#     "question": "Who is Rama?",
#     "context": "Rama (/ˈrɑːmə/;[4] Sanskrit: राम, IAST: Rāma, Sanskrit: [ˈraːmɐ] ⓘ) is a major deity in Hinduism. He is worshipped as the seventh and one of the most popular avatars of Vishnu.[5] In Rama-centric Hindu traditions, he is considered the Supreme Being. Also considered as the ideal man (maryāda puruṣottama), Rama is the male protagonist of the Hindu epic Ramayana. His birth is celebrated every year on Rama Navami, which falls on the ninth day of the bright half (Shukla Paksha) of the lunar cycle of Chaitra (March–April), the first month in the Hindu calendar.[6][7]",
# }

entry = {"question": "Who did not ate the apple?", 
        "context": "Jodi ate banana and Nandu ate apple"}

entry["context"] = clean_text(entry["context"])
entry["question"] = clean_text(entry["question"])

input_text = format_input(entry)
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer).to(device),
    max_new_tokens=50,
    context_size=1024,
    top_k=30,
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)
response_text = generated_text[len(input_text) :].strip()

# print(response_text.strip())
# print("----\n")
print(generated_text)

Below is a question and a context. Your task is to generate an accurate response based on the provided context.

### Question:
Who did not ate the apple?

### Context:
Jodi ate banana and Nandu ate apple

### Response:
Nandu ate banana and Jodi ate apple


In [ ]:
# torch.save(
#     {
#         "model_state_dict": model.state_dict(),
#     },
#     f"Checkpoint/summary1.pth",
# )